# 05. Inference

The saved best model on new texts it has never seen. Nothing is trained here: the model comes from `models/`, written by `make train-improved`.

In [1]:
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd

from itg_classification.config import TEST_PATH
from itg_classification.predict import (
    load_model,
    predict,
    predict_file,
    predictions_table,
)
from itg_classification.split import load_split

pd.set_option("display.max_colwidth", 90)

## 1. Load the saved model

In [2]:
model = load_model()
model.card

{'name': 'LogReg balanced, words + chars',
 'labels': ['negative', 'neutral', 'positive'],
 'trained_at': '2026-10-15T01:23:30+00:00',
 'train_rows': 3842,
 'sklearn_version': '1.9.1',
 'cv_macro_f1': 0.7592744220679827,
 'test_macro_f1': 0.7479431966684092,
 'test_accuracy': 0.8158168574401665}

`load_model` checked that the file matches the code: the same three classes in the same order and the same scikit-learn version.

## 2. New texts

Written by hand, none of them is in the dataset. `expected` is my reading of the sentiment.

In [3]:
cases = pd.DataFrame(
    [
        (
            "news",
            "positive",
            "Nokia raised its full-year sales forecast after a strong third quarter.",
        ),
        (
            "news",
            "positive",
            "The company won a EUR 50 million contract to supply turbines.",
        ),
        (
            "news",
            "negative",
            "Operating profit fell to EUR 3.1 million from EUR 7.8 million a year earlier.",
        ),
        (
            "news",
            "negative",
            "The company will cut 300 jobs as demand weakens.",
        ),
        ("news", "negative", "Shares plunged 15% after the profit warning."),
        (
            "news",
            "neutral",
            "The annual general meeting will be held on 15 March in Helsinki.",
        ),
        (
            "news",
            "neutral",
            "The company has offices in Finland, Sweden and Norway.",
        ),
        ("tweet", "positive", "$AAPL breaking out to new highs, going long"),
        ("tweet", "negative", "$TSLA getting crushed today, glad I sold"),
        ("tweet", "negative", "$NFLX looks weak, adding to my short"),
        ("mixed", "neutral", "Sales rose in Sweden but fell in Finland."),
        (
            "negation",
            "positive",
            "Results were not as bad as analysts feared.",
        ),
        (
            "business event",
            "positive",
            "The group will open 20 new stores next year.",
        ),
    ],
    columns=["kind", "expected", "text"],
)
table = predictions_table(predict(cases["text"].tolist(), model))
results = cases.assign(
    predicted=table["label"],
    confidence=table["confidence"].round(2),
    low_confidence=table["low_confidence"],
    correct=table["label"] == cases["expected"],
)
results

,kind,expected,text,predicted,confidence,low_confidence,correct
0,news,positive,Nokia raised its full-year sales forecast after a strong third quarter.,positive,0.90,False,True
1,news,positive,The company won a EUR 50 million contract to supply turbines.,positive,0.72,False,True
2,news,negative,Operating profit fell to EUR 3.1 million from EUR 7.8 million a year earlier.,positive,0.92,False,False
3,news,negative,The company will cut 300 jobs as demand weakens.,neutral,0.68,False,False
4,news,negative,Shares plunged 15% after the profit warning.,positive,0.46,True,False
5,news,neutral,The annual general meeting will be held on 15 March in Helsinki.,neutral,0.95,False,True
6,news,neutral,"The company has offices in Finland, Sweden and Norway.",neutral,0.88,False,True
7,tweet,positive,"$AAPL breaking out to new highs, going long",positive,0.96,False,True
8,tweet,negative,"$TSLA getting crushed today, glad I sold",negative,0.70,False,True
9,tweet,negative,"$NFLX looks weak, adding to my short",negative,0.90,False,True


In [4]:
results.groupby("correct")["low_confidence"].agg(["size", "sum"])

,size,sum
correct,,
False,6,3
True,7,0


- Right: positive news, neutral news and all three tweets.
- Wrong, and it matches the failure cases of day 5:
  - **negative news**: *operating profit fell* is `positive` with 0.92 confidence, *will cut 300 jobs* is `neutral`. The train set has almost no negative news after day 2;
  - **negation**: *not as bad as feared* is `neutral`;
  - **mixed sentence** and **business event**: both flagged `low_confidence`.
- 3 of the 6 errors are flagged `low_confidence`. The confident ones are negative news and negation, the known blind spots.

## 3. Edge cases

In [5]:
edge_cases = [
    "",
    "   ",
    "http://stks.co/abc",
    "12345 678",
    "RT @trader $AAPL breaking out!!! #bullish https://t.co/x",
    "Le bénéfice a fortement augmenté",
    "profit rose " * 300,
]
edge = predictions_table(predict(edge_cases, model))
edge.assign(text=edge["text"].str.slice(0, 60))[
    ["text", "label", "confidence", "low_confidence", "error"]
].round(2)

,text,label,confidence,low_confidence,error
0,,NaN,NaN,False,no words to classify after preprocessing
1,,NaN,NaN,False,no words to classify after preprocessing
2,http://stks.co/abc,NaN,NaN,False,no words to classify after preprocessing
3,12345 678,NaN,NaN,False,no words to classify after preprocessing
4,RT @trader $AAPL breaking out!!! #bullish https://t.co/x,positive,0.90,False,NaN
5,Le bénéfice a fortement augmenté,neutral,0.62,False,NaN
6,profit rose profit rose profit rose profit rose profit rose,positive,0.94,False,NaN


- Empty text, spaces, a bare link and only numbers get no label: there are no words left after preprocessing. Texts without letters were dropped from training on day 2, so inference rejects them too.
- The retweet with a mention, a link and a hashtag is cleaned and classified as usual.
- A French sentence gets a label: the model cannot tell that a text is not English. It is out of scope, the label means nothing.
- A very long text works: TF-IDF does not limit the length.

## 4. Command line

The same model from a terminal, in a separate process:

In [6]:
command = [sys.executable, "-m", "itg_classification.predict"]
texts = ["Sales rose 20% year-on-year", "$NFLX looks weak"]
print(
    subprocess.run([*command, *texts], capture_output=True, text=True).stdout
)

positive  0.94  Sales rose 20% year-on-year
negative  0.72  $NFLX looks weak



In [7]:
output = subprocess.run(
    [*command, "--json", "The board will meet on Monday"],
    capture_output=True,
    text=True,
)
print(output.stdout)

{"text": "The board will meet on Monday", "label": "neutral", "confidence": 0.7873671054467556, "probabilities": {"negative": 0.05399679657067582, "neutral": 0.7873671054467556, "positive": 0.15863609798256845}, "low_confidence": false, "error": null}



## 5. A CSV file

In [8]:
with tempfile.TemporaryDirectory() as folder:
    input_path = Path(folder) / "news.csv"
    pd.DataFrame({"headline": cases["text"].head(4)}).to_csv(
        input_path, index=False
    )
    saved = predict_file(
        input_path, Path(folder) / "predictions.csv", model, column="headline"
    )
saved.round(2)

,text,label,confidence,p_negative,p_neutral,p_positive,low_confidence,error
0,Nokia raised its full-year sales forecast after a strong third quarter.,positive,0.90,0.06,0.04,0.90,False,None
1,The company won a EUR 50 million contract to supply turbines.,positive,0.72,0.02,0.26,0.72,False,None
2,Operating profit fell to EUR 3.1 million from EUR 7.8 million a year earlier.,positive,0.92,0.01,0.07,0.92,False,None
3,The company will cut 300 jobs as demand weakens.,neutral,0.68,0.19,0.68,0.13,False,None


## 6. Speed

In [9]:
texts = load_split(TEST_PATH)["raw_text"].tolist()
start = time.perf_counter()
predict(texts, model)
seconds = time.perf_counter() - start
f"{len(texts)} texts in {seconds:.2f} s"

'961 texts in 0.07 s'

Preprocessing and prediction of the whole test set take a fraction of a second, with no training.

## 7. Summary

- The model is loaded from disk and checked against its model card, nothing is retrained.
- New texts go through the same `preprocess_text` as the training data; texts without words get an error instead of a label.
- The answers repeat the error analysis: news with growth, neutral news and tweets are right, negative news and negation are not.
- `low_confidence` catches most of the doubtful answers, but not the confident mistakes on negative news.